In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
fert_total = pd.read_csv('data/fert-total-1950-1982.txt', dtype={"STCOFIPS": str}).set_index('fips-int')
fert_farm = pd.read_csv('data/fert-farm-1987-2017.txt', dtype={"STCOFIPS": str}).set_index('fips-int')
fert_nonfarm = pd.read_csv('data/fert-nonfarm-1987-2017.txt', dtype={"STCOFIPS": str}).set_index('fips-int')
county_data_2022 = pd.read_csv('../data/county_data_2022.csv', dtype={"FIPS": str})


In [ ]:
county_data_2022 = county_data_2022.rename(columns={'FIPS':'STCOFIPS'})

In [ ]:
county_data_2022

In [ ]:
# Remove phosphorus data

fert_total = fert_total.iloc[:, :11]
fert_farm = fert_farm.iloc[:, :10]
fert_nonfarm = fert_nonfarm.iloc[:, :10]


In [ ]:
# This code is overkill because the datasets have the same indices, but good practice I think.

common_index = fert_farm.index.intersection(fert_nonfarm.index)

fert_farm = fert_farm.loc[common_index]
fert_nonfarm = fert_nonfarm.loc[common_index]

for year in range(1987, 2022, 5):
    fert_total[f'tot-fertN-kg-{year}'] = (
        fert_farm[f'farmfertN-kg-{year}'] +
        fert_nonfarm[f'nonffertN-kg-{year}']
    )



In [ ]:
fert_farm = pd.merge(fert_farm, county_data_2022, on='STCOFIPS')

fert_farm

In [ ]:
for year in range(1987, 2022, 5):
    
    fert_farm[f"nitrogen_kg_km_sq_{year}"] = fert_farm[f'farmfertN-kg-{year}'] / fert_farm['area_land_km2']
    
fert_farm

### Create a map of county level fertilizer levels for a given year

In [ ]:
import requests

url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

counties_geojson = requests.get(url).json()

In [ ]:
# import plotly.express as px

# fig = px.choropleth(
#     fert_farm, 
#     geojson=counties_geojson, 
#     locations='STCOFIPS', 
#     color='farmfertN-kg-2017',
#     color_continuous_scale="Viridis",
#     scope="usa"
# )
# fig.show()

In [ ]:
years = range(1987, 2022, 5)

nitrogen_long = fert_farm.melt(
    id_vars=["STCOFIPS", "CountyName", "State"],
    value_vars=[f"nitrogen_kg_km_sq_{year}" for year in years],
    var_name="year",
    value_name="nitrogen_kg_km_sq"
)

nitrogen_long["year"] = (
    nitrogen_long["year"]
    .str.extract(r"(\d+)")
    .astype(int)
)

nitrogen_long["STCOFIPS"] = nitrogen_long["STCOFIPS"].astype(str)

# nitrogen_long["log_nitrogen"] = np.log1p(
#     nitrogen_long["nitrogen"]
# )

vmin = nitrogen_long["nitrogen_kg_km_sq"].min()
vmax = nitrogen_long["nitrogen_kg_km_sq"].max()

nitrogen_long

In [ ]:
# for year in range(1987, 2022, 5):

#     fig = px.choropleth(
#         fert_farm, 
#         geojson=counties_geojson, 
#         locations='STCOFIPS', 
#         color=f"log_nitrogen_{year}",
#         range_color=(vmin, vmax),
#         color_continuous_scale="Viridis",
#         scope="usa"
#     )
#     fig.show()

import plotly.express as px

fig = px.choropleth(
    nitrogen_long,
    geojson=counties_geojson,
    locations="STCOFIPS",
    color="nitrogen_kg_km_sq",
    animation_frame="year",
    scope="usa",
    range_color=(vmin, vmax),
    hover_name="CountyName",
    hover_data=["State", "nitrogen_kg_km_sq"],
    title="County Nitrogen Fertilizer Application Over Time"
)

fig.update_geos(
    visible=False
)

fig.show()    

